# SHAP — A Unified Approach to Interpreting Model Predictions (2017)

_Papers · Meta-learning, Bayesian & Robustness_

**Attribute a prediction to its features with the unique fair split borrowed from cooperative game theory.**

---

We compute exact SHAP values by enumerating every coalition of features, verify they match the closed-form ground truth on an additive model, and then break the factorial weighting to see exactly which guarantee it protects. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Compute exact Shapley values by enumerating coalitions

A SHAP value answers: how much does feature `i` contribute to this prediction, fairly averaged over every order in which features could be added? The exact formula (Eqn. 8) enumerates every coalition `S` of the *other* features and adds feature `i`'s marginal contribution `value(S ∪ {i}) - value(S)`, weighted by `|S|!(M-|S|-1)!/M!`. We code that directly, then replay the 2-feature worked example and confirm the values sum to `f(x) - E[f]`.

In [ ]:
import itertools
import math
import numpy as np
import torch

# ---- Recompute the worked example: 2-feature additive model, exact Shapley. ----
def shapley(value_fn, F):
    """Exact Shapley value (SHAP, Eqn. 8): enumerate every coalition of the OTHER features."""
    M = len(F)
    phi = {}
    for i in F:
        rest = [j for j in F if j != i]
        total = 0.0
        for r in range(len(rest) + 1):
            for S in itertools.combinations(rest, r):       # every subset S of F\{i}
                S = frozenset(S)
                w = math.factorial(len(S)) * math.factorial(M - len(S) - 1) / math.factorial(M)
                total += w * (value_fn(S | {i}) - value_fn(S))  # weighted marginal contribution
        phi[i] = total
    return phi

base2 = 10.0
c2 = {0: 3.0, 1: 4.0}
x2 = {0: 2.0, 1: 5.0}
mean2 = {0: 0.0, 1: 0.0}
F2 = [0, 1]

def value2(S):                                              # f_x(S): present at x, absent at mean
    return base2 + sum(c2[i] * (x2[i] if i in S else mean2[i]) for i in F2)

phi2 = shapley(value2, F2)
fx2 = value2(frozenset(F2))                                 # full prediction
Ef2 = value2(frozenset())                                   # base value E[f]
print("worked example (2 feat): phi =", {k: round(v, 4) for k, v in phi2.items()})
print("  base phi0 = E[f] =", round(Ef2, 4), " f(x) =", round(fx2, 4))
print("  sum(phi) =", round(sum(phi2.values()), 4), " == f(x)-E[f] =", round(fx2 - Ef2, 4))
print("  phi0 + sum(phi) =", round(Ef2 + sum(phi2.values()), 4), " == f(x) =", round(fx2, 4))
# worked example (2 feat): phi = {0: 6.0, 1: 20.0}
#   base phi0 = E[f] = 10.0  f(x) = 36.0
#   sum(phi) = 26.0  == f(x)-E[f] = 26.0
#   phi0 + sum(phi) = 36.0  == f(x) = 36.0

### Step 2 — Check the values against a known ground truth

For a purely additive model `f(x) = bias + W·x`, the correct attribution for feature `i` is known in closed form: `c_i * (x_i - mean_i)`. We build such a model with torch, define the coalition value function (present features sit at `x`, absent ones at the background `mean`), and confirm the exact SHAP values reproduce that ground truth — along with **local accuracy** (`phi0 + sum(phi) = f(x)`) and **efficiency** (`sum(phi) = f(x) - E[f]`).

In [ ]:
# ---- A 3-feature additive model composed with torch; ground truth is c_i*(x_i - mean_i). ----
W = torch.tensor([2.0, -3.0, 0.5])      # per-feature coefficients
bias = torch.tensor(1.0)

def model(xt):
    return (xt @ W) + bias              # additive: f(x) = bias + W . x

x = torch.tensor([4.0, 2.0, 6.0])       # the instance to explain
mean = torch.tensor([1.0, 1.0, 1.0])    # background means -> absent features use these
F = [0, 1, 2]

def value(S):                           # f_x(S): present features at x, absent at the mean
    present = torch.tensor([i in S for i in F])
    xt = torch.where(present, x, mean)
    return model(xt).item()

phi = shapley(value, F)
fx = value(frozenset(F))
Ef = value(frozenset())
ground = {i: (W[i] * (x[i] - mean[i])).item() for i in F}   # additive ground truth

print("\n3-feature additive model — exact SHAP values:")
for i in F:
    print("  phi[%d] = % .4f   ground truth c_i*(x_i-mean_i) = % .4f" % (i, phi[i], ground[i]))
print("  match ground truth:", np.allclose([phi[i] for i in F], [ground[i] for i in F]))
print("  base phi0 = E[f] = %.4f   f(x) = %.4f" % (Ef, fx))
print("  LOCAL ACCURACY: phi0 + sum(phi) = %.4f  == f(x) = %.4f  -> %s" % (
      Ef + sum(phi.values()), fx, np.isclose(Ef + sum(phi.values()), fx)))
print("  EFFICIENCY:     sum(phi) = %.4f  == f(x)-E[f] = %.4f" % (sum(phi.values()), fx - Ef))
# 3-feature additive model — exact SHAP values:
#   phi[0] =  6.0000   ground truth ... =  6.0000
#   phi[1] = -3.0000   ground truth ... = -3.0000
#   phi[2] =  2.5000   ground truth ... =  2.5000
#   match ground truth: True
#   base phi0 = E[f] = 0.5000   f(x) = 6.0000
#   LOCAL ACCURACY: phi0 + sum(phi) = 6.0000  == f(x) = 6.0000  -> True
#   EFFICIENCY:     sum(phi) = 5.5000  == f(x)-E[f] = 5.5000

### Step 3 — Ablate the factorial weights and watch a guarantee break

Why the funny `|S|!(M-|S|-1)!/M!` weight? Those factorials are the order-counts that make the marginal contributions telescope to `f(x) - E[f]` when summed. Swap them for a flat `1/2^(M-1)` that treats every coalition equally and the per-feature numbers may still look plausible, but the sum no longer reconstructs the prediction — local accuracy / efficiency fails, which the check below detects.

In [ ]:
# ---- ABLATION: drop the factorial weights (flat 1/2^(M-1) per coalition) -> local accuracy breaks. ----
def shapley_flat(value_fn, F):
    M = len(F)
    phi = {}
    for i in F:
        rest = [j for j in F if j != i]
        total = 0.0
        for r in range(len(rest) + 1):
            for S in itertools.combinations(rest, r):
                S = frozenset(S)
                total += (1.0 / 2 ** (M - 1)) * (value_fn(S | {i}) - value_fn(S))   # WRONG weight
        phi[i] = total
    return phi

phi_flat = shapley_flat(value, F)
print("\nABLATION (flat weights, no factorials):")
print("  sum(phi_flat) = %.4f   should equal f(x)-E[f] = %.4f  -> %s" % (
      sum(phi_flat.values()), fx - Ef, np.isclose(sum(phi_flat.values()), fx - Ef)))
# ABLATION: sum(phi_flat) != f(x)-E[f]  -> local accuracy / efficiency is broken without the factorial weights.

## Visualize the data & results

_Do the exact SHAP values reconstruct the prediction? Each bar is a feature's attribution; stacked on the base value E[f] they must reach f(x) (local accuracy, Eqn. 5)._

### Step 1 — Recompute the values for the bar chart

This self-contained block re-defines the exact `shapley` function and the same additive model, then computes the base value `E[f]`, the per-feature attributions `phi`, and the full prediction `f(x)`. These are exactly the quantities the stacked bar chart draws.

In [ ]:
import itertools
import math
import numpy as np
import torch

def shapley(value_fn, F):                       # exact SHAP value, Eqn. 8
    M = len(F)
    phi = {}
    for i in F:
        rest = [j for j in F if j != i]
        total = 0.0
        for r in range(len(rest) + 1):
            for S in itertools.combinations(rest, r):
                S = frozenset(S)
                w = math.factorial(len(S)) * math.factorial(M - len(S) - 1) / math.factorial(M)
                total += w * (value_fn(S | {i}) - value_fn(S))
        phi[i] = total
    return phi

W = torch.tensor([2.0, -3.0, 0.5])
bias = torch.tensor(1.0)
x = torch.tensor([4.0, 2.0, 6.0])
mean = torch.tensor([1.0, 1.0, 1.0])
F = [0, 1, 2]

def model(xt):
    return (xt @ W) + bias

def value(S):
    xt = torch.where(torch.tensor([i in S for i in F]), x, mean)
    return model(xt).item()

phi = shapley(value, F)
fx = value(frozenset(F))
Ef = value(frozenset())

### Step 2 — Confirm the bars stack from E[f] up to f(x)

The point of the chart is the **local accuracy** identity: starting from the base value `E[f]` and stacking each feature's attribution must land exactly on `f(x)` (Eqn. 5). We print all the pieces so you can see `E[f] + sum(phi) = f(x)` and `sum(phi) = f(x) - E[f]` hold to the digit.

In [ ]:
print("base E[f] =", round(Ef, 4))
print("phi       =", [round(phi[i], 4) for i in F])
print("f(x)      =", round(fx, 4))
print("E[f] + sum(phi) =", round(Ef + sum(phi.values()), 4), " (local accuracy: equals f(x))")
print("sum(phi)        =", round(sum(phi.values()), 4), " = f(x)-E[f] =", round(fx - Ef, 4))
# base E[f] = 0.5
# phi       = [6.0, -3.0, 2.5]
# f(x)      = 6.0
# E[f] + sum(phi) = 6.0   (local accuracy: equals f(x))
# sum(phi)        = 5.5   = f(x)-E[f] = 5.5
# Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The weighting ablation. You have a working exact Shapley computation. Replace the factorial
            coalition weight $\tfrac{|S|!\,(M-|S|-1)!}{M!}$ with a flat weight $\tfrac{1}{2^{M-1}}$
            (every coalition counted equally) and recompute. What property breaks, and how do you detect it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Swap the weight: in the inner loop change w = factorial(|S|)*factorial(M-|S|-1)/factorial(M) to w = 1/2**(M-1). — _A flat weight treats all coalitions as equally likely, ignoring that more orderings produce some coalitions than others._
- Recompute $\phi_i$ and check the sum: compare $\phi_0 + \sum_i \phi_i$ against $f(x)$. — _Local accuracy (Eqn. 5) is the telescoping identity; it relies on the factorial weights summing per-order to 1. Flat weights break the telescoping._
- Observe that for the symmetric additive case the per-feature numbers may still look plausible, but the sum no longer equals $f(x) - E[f]$ in general. — _The ablation keeps each marginal but mis-weights them, so efficiency &mdash; the guarantee SHAP is built on &mdash; is lost._

**Answer:** You break local accuracy / efficiency (Property 1, Eqn. 5). The factorial weights are
                 exactly the order-counts that make the marginals telescope to $f(x) - E[f]$ when summed; replace
                 them with a flat $\tfrac{1}{2^{M-1}}$ and the attributions no longer add up to the prediction minus
                 the base value. You detect it by the failing check $\phi_0 + \sum_i \phi_i \ne f(x)$. This is why
                 the weighting is not optional &mdash; it is what makes SHAP values the fair split rather than
                 just an average of marginals. (Our run prints both sums so the gap is visible.)

</details>

**Problem 2.** Take the additive model from the example, $f(x) = 10 + 3x_0 + 4x_1$, but now explain $x = (1, -2)$ with
            background means $\bar x = (0, 0)$. Compute $\phi_0$, $\phi_0$ via the base value, $\phi_1$, and verify
            local accuracy by hand.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Value function: $f_x(\varnothing)=10$, $f_x(\{0\})=10+3\cdot1=13$, $f_x(\{1\})=10+4\cdot(-2)=2$, $f_x(\{0,1\})=10+3-8=5$. Base value $\phi_0=10$, prediction $f(x)=5$. — _Absent features sit at the mean $0$; present ones at $x$. This defines every coalition's payout._
- Feature 0 marginals: $13-10=3$ and $5-2=3$; weights $\tfrac12,\tfrac12$, so $\phi_0^{(\text{feat})}=3$. Feature 1 marginals: $2-10=-8$ and $5-13=-8$; so $\phi_1=-8$. — _For an additive model the marginal is constant across coalitions: $c_i(x_i-\bar x_i)$. Here $3(1)=3$ and $4(-2)=-8$._
- Check: $10 + 3 + (-8) = 5 = f(x)$, and $\sum_i\phi_i = -5 = f(x)-E[f] = 5-10$. — _Local accuracy: base value plus attributions equals the prediction; the attributions equal prediction minus base._

**Answer:** $\phi_0 (\text{base}) = 10$, $\phi_{\text{feat }0} = 3$, $\phi_{\text{feat }1} = -8$. Feature 1
                 is now negative &mdash; it pulled the prediction down because its value $-2$ is below its mean
                 $0$. Local accuracy holds: $10 + 3 - 8 = 5 = f(x)$, and $\sum_i\phi_i = -5 = f(x)-E[f]$. The signs
                 read off directly: a feature above its mean (feature 0) contributes positively here, one below its
                 mean (feature 1) contributes negatively, each scaled by its coefficient.

</details>

**Problem 3.** Why do SHAP values sum to $f(x) - E[f]$ rather than to $f(x)$, and why is that the right thing for
            an explanation?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the base value $\phi_0 = f_x(\varnothing) = E[f]$: the output when no feature is known, i.e. the average prediction over the background. — _An explanation needs a reference point &mdash; "compared to knowing nothing." That reference is $E[f]$, not $0$._
- Apply local accuracy (Eqn. 5): $f(x) = \phi_0 + \sum_i \phi_i$, hence $\sum_i \phi_i = f(x) - \phi_0 = f(x) - E[f]$. — _The attributions measure movement away from the baseline, so they must sum to the gap between this prediction and the baseline._
- Interpret: each $\phi_i$ is "how much knowing feature $i$ shifted us from the average prediction toward this specific one." — _That is exactly what a per-feature explanation should answer; summing them must therefore reconstruct the total shift $f(x)-E[f]$._

**Answer:** Because attributions explain movement from a baseline, and the baseline is the
                 no-information prediction $\phi_0 = E[f]$, not zero. Local accuracy gives
                 $f(x) = \phi_0 + \sum_i \phi_i$, so $\sum_i \phi_i = f(x) - E[f]$. This is the right target: a
                 feature's SHAP value answers "how much did knowing this feature move us from the average prediction
                 to this one?", and those movements must add up to the total gap $f(x)-E[f]$. Expecting them to sum to
                 $f(x)$ ignores that you start from $E[f]$, not from nothing.

</details>